In [17]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    train_path: Path = Path("../data/train.csv")
    test_path: Path = Path("../data/test.csv")
    sub_path: Path = Path("../data/sample_submission.csv")
    pltpd_path: Path = Path("../data/podcast_dataset.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('../data/train.csv'),
 'test_path': PosixPath('../data/test.csv'),
 'sub_path': PosixPath('../data/sample_submission.csv'),
 'pltpd_path': PosixPath('../data/podcast_dataset.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.02,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    df.loc[df['Episode_Length_minutes']>121.0, 'Episode_Length_minutes'] = 121.0

    df['Host_Guest_Diff'] = df['Host_Popularity_percentage'] - df['Guest_Popularity_percentage']
    df['Host_Guest_Ratio'] = (df['Host_Popularity_percentage'] / df['Guest_Popularity_percentage']).replace([float('inf'), -float('inf')], pd.NA)

    if "Listening_Time_minutes" in df.columns:
        df['Listening_Episode_Diff'] = df['Episode_Length_minutes'] - df['Listening_Time_minutes']
        df['Listening_Episode_Ratio'] = (df['Episode_Length_minutes'] / df['Listening_Time_minutes']).replace([float('inf'), -float('inf')], pd.NA)

    return df


df_train = pd.read_csv(cfg.train_path)
df_test = pd.read_csv(cfg.test_path)
df_sub = pd.read_csv(cfg.sub_path)
df_pltpd = pd.read_csv(cfg.pltpd_path)

df_pltpd = df_pltpd.dropna(subset=['Listening_Time_minutes'])
df_pltpd = df_pltpd.reset_index(drop=True)
df_pltpd.index = df_pltpd.index + 1000000
df_pltpd['id'] = df_pltpd.index + 1000000

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)
df_pltpd = preprocess_df(df_pltpd)

df_comb = pd.concat([df_train, df_pltpd], axis=0)
df_comb = df_comb.reset_index(drop=True)

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

display(df_comb)
display(df_comb.describe())

,id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
0,0,0,NaN,0,74.81,3,21,NaN,0.0,2,31.419980,98,NaN,NaN,NaN,NaN
1,1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.012410,26,-9.00,0.881501,31.787590,1.361172
2,2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.925310,16,61.00,7.800446,28.974690,1.644952
3,3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.278240,45,-21.48,0.727065,20.891760,1.451438
4,4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.610310,86,21.39,1.364519,34.899690,1.461573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797100,1047100,33,24.81,9,66.15,0,17,98.63,1.0,1,20.573795,17,-32.48,0.670688,4.236205,1.205903
797101,1047101,11,92.15,6,89.61,5,21,25.82,2.0,0,76.198459,9,63.79,3.470565,15.951541,1.209342
797102,1047102,23,112.27,1,26.33,5,21,55.29,0.0,1,107.602135,24,-28.96,0.476216,4.667865,1.043381
797103,1047103,19,NaN,8,41.47,2,14,33.58,0.0,1,17.220998,85,7.89,1.234961,NaN,NaN


,id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Listening_Episode_Diff
count,7.971050e+05,797105.000000,705317.000000,797105.000000,797105.000000,797105.000000,797105.000000,646356.000000,797104.000000,797105.000000,797105.000000,797105.000000,646356.000000,705317.000000
mean,4.133258e+05,23.540988,64.408705,4.554814,59.877839,3.028731,15.663856,52.095246,1.357792,0.998145,45.444668,51.378954,7.627507,18.679341
std,2.598146e+05,13.911304,32.981409,2.962341,22.889880,2.022848,4.027274,28.483819,1.149681,0.815531,27.140915,28.131239,36.152160,13.566281
min,0.000000e+00,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-80.170000,-115.540000
25%,1.992760e+05,12.000000,35.670000,2.000000,39.450000,1.000000,14.000000,28.100000,0.000000,0.000000,23.184220,28.000000,-18.280000,8.130000
50%,3.985520e+05,23.000000,63.770000,5.000000,60.060000,3.000000,17.000000,53.350000,1.000000,1.000000,43.392270,52.000000,6.640000,15.643750
75%,5.978280e+05,36.000000,94.000000,7.000000,79.560000,5.000000,21.000000,76.490000,2.000000,2.000000,64.814620,75.000000,33.000000,26.683090
max,1.047104e+06,47.000000,121.000000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,119.970000,100.000000,113.550000,103.220440


In [41]:
df_dup = df_comb.copy()
df_dup = df_dup.dropna(subset=['Guest_Popularity_percentage'])
df_dup = df_dup[df_dup.duplicated(subset=['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'], keep=False)]
# df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup = df_dup.sort_values(['Listening_Time_minutes'])
df_dup

,id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio
794928,1044928,17,6.50,1,26.83,2,21,46.69,3.0,1,0.00000,32,-19.86,0.574641,6.50000,<NA>
751352,1001352,23,5.67,1,35.97,4,17,97.55,2.0,2,0.00000,38,-61.58,0.368734,5.67000,<NA>
750348,1000348,26,7.76,7,81.33,6,17,92.41,3.0,2,0.00000,76,-11.08,0.8801,7.76000,<NA>
796316,1046316,43,11.02,1,85.23,3,14,5.79,3.0,1,0.00000,73,79.44,14.720207,11.02000,<NA>
786679,1036679,7,NaN,5,47.26,1,14,96.43,3.0,0,0.00000,72,-49.17,0.490096,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361478,361478,41,NaN,5,83.55,6,10,54.69,3.0,2,119.10467,46,28.86,1.527702,NaN,NaN
109485,109485,41,119.17,5,83.55,6,10,54.69,3.0,2,119.10467,46,28.86,1.527702,0.06533,1.000549
87034,87034,15,120.32,3,78.39,3,10,54.64,1.0,1,119.66000,31,23.75,1.434663,0.66000,1.005516
181281,181281,15,118.19,3,78.39,1,10,54.64,1.0,1,119.66000,31,23.75,1.434663,-1.47000,0.987715


In [42]:
cols_to_compare = ['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage']

df_test_with_id = df_comb.copy()
df_test_with_id = df_test_with_id.dropna(subset=['Guest_Popularity_percentage'])

leaked_rows = df_test_with_id.merge(
    df_train[cols_to_compare + ['Listening_Time_minutes']].drop_duplicates(),
    on=cols_to_compare,
    how='inner'
)

mean_values = leaked_rows.groupby('id')['Listening_Time_minutes'].mean().reset_index()

display(df_test.loc[df_test['id'].isin(mean_values['id'].values)])
df_test.loc[df_test['id'].isin(mean_values['id'].values), "Listening_Time_minutes2"] = mean_values["Listening_Time_minutes"].values
display(df_test.loc[df_test['id'].isin(mean_values['id'].values)])

KeyError: 'Column not found: Listening_Time_minutes'

In [43]:
leaked_rows

,id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes_x,Episode_Num,Host_Guest_Diff,Host_Guest_Ratio,Listening_Episode_Diff,Listening_Episode_Ratio,Listening_Time_minutes_y
0,1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.012410,26,-9.00,0.881501,31.787590,1.361172,88.01241
1,2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.925310,16,61.00,7.800446,28.974690,1.644952,44.92531
2,3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.278240,45,-21.48,0.727065,20.891760,1.451438,46.27824
3,4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.610310,86,21.39,1.364519,34.899690,1.461573,75.61031
4,6,6,69.83,0,35.82,6,21,39.02,0.0,1,64.750240,47,-3.20,0.917991,5.079760,1.078452,64.75024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608777,1047064,32,104.55,4,25.84,3,10,88.89,0.0,1,84.539392,37,-63.05,0.290696,20.010608,1.236702,84.53939
608778,1047066,44,19.15,0,69.81,5,10,0.70,3.0,2,18.695367,51,69.11,99.728571,0.454633,1.024318,18.69537
608779,1047070,19,NaN,8,74.49,3,21,64.94,1.0,2,24.419968,86,9.55,1.147059,NaN,NaN,24.41997
608780,1047073,5,54.63,4,35.80,2,21,88.80,0.0,2,53.978371,25,-53.00,0.403153,0.651629,1.012072,53.97837
